In [5]:
from pathlib import Path
import json
import time
import urllib.request
import pandas as pd
import torch
import transformers
import sentencepiece

PROJECT_DIR = Path("..")

# European Parliament data
EP_DATA_DIR = PROJECT_DIR / "EP" / "EP-data"

# Danish Parliament data
DK_DATA_DIR = PROJECT_DIR / "data" / "temp_data" / "parliament"

# All period folders for summary
PERIOD_FOLDERS = ["2009-2014", "2014-2019", "2019-2024", "2024-2029"]

In [6]:
total_rows = 0

for folder_name in PERIOD_FOLDERS:
    path = EP_DATA_DIR / folder_name / "ep_votes.csv"

    if not path.exists():
        print(f"{folder_name}: ep_votes.csv NOT FOUND")
        continue

    df = pd.read_csv(path)
    n_rows = len(df)

    print(f"{folder_name}: {n_rows:,} rows")

    total_rows += n_rows

print(f"\nTotal: {total_rows:,} rows")

2009-2014: 6,961 rows
2014-2019: 10,252 rows
2019-2024: 1,808 rows
2024-2029: 614 rows

Total: 19,635 rows


In [7]:
roll_calls = pd.read_csv(DK_DATA_DIR / "roll_calls.csv")

print(f"Number of Danish roll calls: {len(roll_calls):,}")

Number of Danish roll calls: 2,128


In [8]:
print(roll_calls.columns.tolist())

['afstemningid', 'sagstrinid', 'kommentar', 'afstemningstypeid', 'dato', 'sagstrintypeid', 'sagid', 'sagstypeid', 'sag_nummer', 'sag_titel', 'sag_titelkort']


In [9]:
roll_calls[["kommentar", "sag_titel", "sag_titelkort"]].head(20)

,kommentar,sag_titel,sag_titelkort
0,NaN,Forslag til lov om ændring af virksomhedsskatt...,Om indgreb mod utilsigtet udnyttelse af virkso...
1,NaN,Forslag til lov om ændring af lov om afgift af...,Om tilbagerulning af forsyningssikkerhedsafgif...
2,NaN,Forslag til lov om ændring af lov om produktio...,Om kompetencebevis.
3,NaN,Forslag til lov om dansk turisme.,Om dansk turisme.
4,NaN,Forslag til lov om ændring af lov om aktiv soc...,Om ændring af formue- og fradragsregler ved ef...
5,NaN,Forslag til lov om ændring af lov om ferie. (F...,Om færre betingelser for optjening af sygeferi...
6,NaN,Forslag til lov om ændring af lov om vikarers ...,Om overførsel af kompetencer til Arbejdsretten.
7,NaN,Forslag til lov om ophævelse af lov om nærings...,Om afskaffelse af næringsbrevsordningen.
8,NaN,Forslag til lov om ændring af konkurrenceloven...,Om ændring af konkurrenceloven m.v.
9,NaN,Forslag til lov om Danmarks Grønne Investering...,Om Danmarks Grønne Investeringsfond.


All "kommentar" s are NaN so we are gonna just ignore them

In [10]:
roll_calls["kommentar"].unique()

array([nan])

In [11]:
roll_calls = roll_calls.drop(columns=["kommentar"])

Almost all are unique so we ll just translate everything

In [12]:
print(f"Total roll calls: {len(roll_calls):,}")
print(f"Unique cases (sagid): {roll_calls['sagid'].nunique():,}")
print(f"Unique full titles: {roll_calls['sag_titel'].nunique():,}")
print(f"Unique short titles: {roll_calls['sag_titelkort'].nunique():,}")

print("\nMissing values:")
print(roll_calls[["sag_titel", "sag_titelkort"]].isna().sum())

Total roll calls: 2,128
Unique cases (sagid): 2,128
Unique full titles: 2,091
Unique short titles: 2,098

Missing values:
sag_titel        0
sag_titelkort    0
dtype: int64


In [13]:
from transformers import MarianMTModel, MarianTokenizer

model_name = "Helsinki-NLP/opus-mt-da-en"

tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)

print("Model loaded!")

tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

c:\Users\kubic\Desktop\machine learning\Danish-politics-project\.venv\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\kubic\.cache\huggingface\hub\models--Helsinki-NLP--opus-mt-da-en. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


source.spm:   0%|          | 0.00/820k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/788k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/300M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

Model loaded!


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/300M [00:00<?, ?B/s]

In [14]:
def translate_texts(texts):
    inputs = tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512
    )

    translated = model.generate(
        **inputs,
        max_length=512
    )

    return tokenizer.batch_decode(
        translated,
        skip_special_tokens=True
    )

In [15]:
test_titles = roll_calls["sag_titel"].head(10).tolist()

translations = translate_texts(test_titles)

for danish, english in zip(test_titles, translations):
    print("DA:", danish)
    print("EN:", english)
    print()

DA: Forslag til lov om ændring af virksomhedsskatteloven og kildeskatteloven. (Indgreb mod utilsigtet udnyttelse af virksomhedsordningen ved indskud af privat gæld m.v.).
EN: Proposal for a law amending the Corporate Tax Act and the Law on withholding tax. (Action against unintended exploitation of the corporate scheme by private debt deposits, etc.).

DA: Forslag til lov om ændring af lov om afgift af elektricitet, lov om afgift af stenkul, brunkul og koks m.v., personskatteloven og forskellige andre love. (Tilbagerulning af forsyningssikkerhedsafgiften, nedsættelse af elvarmeafgiften, forhøjelse af elafgiften, tilpasning af afgiftsregler for VE-brændsler, forhøjelse af bundskatten og det skrå skatteloft, nedsættelse af den grønne check og afgiftsforhøjelse på cigarillos m.v.).
EN: Proposal for a law amending the Law on the tax of electricity, the Law on the tax of coal, lignite and coke, the Act on the tax of persons and various other laws (Rewinding of the security of supply tax, re

Createing a 100 rows sample for manual translation checking

In [16]:
translation_sample = (
    roll_calls[["afstemningid", "sag_titel"]]
    .sample(n=100, random_state=42)
    .copy()
)

translation_sample["sag_titel_en"] = translate_texts(
    translation_sample["sag_titel"].tolist()
)

translation_sample.head()

,afstemningid,sag_titel,sag_titel_en
282,3066,Forslag til lov om ændring af straffeloven. (S...,Draft law amending the penal code.
2000,10258,Forslag til lov om ændring af lov om social pe...,Proposal for a law amending the Social Pension...
1706,9369,Forslag til lov om ændring af personskattelove...,Proposal for a law amending the Personal Tax A...
988,7198,Forslag til lov om ændring af landbrugsstøttel...,Proposal for a law amending the Agricultural S...
2019,10325,Forslag til lov om ændring af lov om Det Centr...,Proposal for a law amending the Central Busine...


In [17]:
translation_sample["translation_correct"] = ""

In [18]:
sample_path = DK_DATA_DIR / "translation_validation_100.csv"

translation_sample.to_csv(
    sample_path,
    index=False
)

print(f"Saved validation sample to: {sample_path}")

Saved validation sample to: ..\data\temp_data\parliament\translation_validation_100.csv


Translating whole dataset

In [19]:
def translate_all_titles(texts, batch_size=16):
    translations = []

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]

        translated_batch = translate_texts(batch)
        translations.extend(translated_batch)

        print(
            f"Translated {min(i + batch_size, len(texts)):,} "
            f"/ {len(texts):,}"
        )

    return translations

In [21]:
roll_calls["sag_titel_en"] = translate_all_titles(
    roll_calls["sag_titel"].tolist(),
    batch_size=100
)

Translated 100 / 2,128
Translated 200 / 2,128
Translated 300 / 2,128
Translated 400 / 2,128
Translated 500 / 2,128
Translated 600 / 2,128
Translated 700 / 2,128
Translated 800 / 2,128
Translated 900 / 2,128
Translated 1,000 / 2,128
Translated 1,100 / 2,128
Translated 1,200 / 2,128
Translated 1,300 / 2,128


KeyboardInterrupt: 

In [ ]:
roll_calls[
    ["sag_titel", "sag_titel_en"]
].sample(20, random_state=42)

In [ ]:
full_path = DK_DATA_DIR / "roll_calls_translated.csv"

roll_calls.to_csv(
    full_path,
    index=False
)

print(f"Saved translated roll calls to: {full_path}")